In [1]:
# Import libraries
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')

# SHAP
import shap
import matplotlib.pyplot as plt
import seaborn as sns

# Plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# Color palette
COLORS = {
    'primary': '#003366',
    'secondary': '#FFB612',
    'success': '#28A745',
    'danger': '#DC3545',
    'warning': '#FFC107',
    'info': '#17A2B8'
}

print("✅ Libraries imported!")


✅ Libraries imported!


In [2]:
# Load data and model
X_train = joblib.load('../data/processed/X_train.pkl')
X_test = joblib.load('../data/processed/X_test.pkl')
y_test = joblib.load('../data/processed/y_test.pkl')

# Load best model
model = joblib.load('../models/best_model.pkl')
optimal_threshold = joblib.load('../models/optimal_threshold.pkl')
feature_names = joblib.load('../models/feature_names.pkl')

print("📊 Data and Model Loaded Successfully!")
print("="*60)
print(f"Model type: {type(model).__name__}")
print(f"Optimal threshold: {optimal_threshold:.3f}")
print(f"Test set: {X_test.shape[0]:,} samples")
print(f"Features: {X_test.shape[1]}")


📊 Data and Model Loaded Successfully!
Model type: RandomForestClassifier
Optimal threshold: 0.160
Test set: 16,000 samples
Features: 100


In [ ]:
# SHAP Explainability - Complete Working Version
print("="*60)
print("🔍 CREATING SHAP EXPLAINER")
print("="*60)

# Check model type
model_type = type(model).__name__
print(f"Model type: {model_type}")

# Get feature names
feature_names = X_train.columns.tolist()
print(f"Number of features: {len(feature_names)}")

# Sample background data for speed
n_background = min(50, len(X_train))
X_background = X_train.sample(n_background, random_state=42)
print(f"Background samples: {n_background}")

# Try SHAP
shap_success = False

# Method 1: TreeExplainer
try:
    print("\nTrying TreeExplainer...")
    explainer = shap.TreeExplainer(model, X_background)
    shap_values = explainer.shap_values(X_test)
    
    # Handle list output
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    
    print("✅ TreeExplainer SUCCESS!")
    shap_success = True
    
except Exception as e:
    print(f"❌ TreeExplainer failed: {str(e)[:100]}")

# Method 2: KernelExplainer (if TreeExplainer failed)
if not shap_success:
    try:
        print("\nTrying KernelExplainer...")
        
        def predict_fn(x):
            return model.predict_proba(x)[:, 1]
        
        explainer = shap.KernelExplainer(predict_fn, X_background)
        shap_values = explainer.shap_values(X_test, nsamples=50)
        
        print("✅ KernelExplainer SUCCESS!")
        shap_success = True
        
    except Exception as e:
        print(f"❌ KernelExplainer failed: {str(e)[:100]}")

# Method 3: LinearExplainer (if others failed)
if not shap_success:
    try:
        print("\nTrying LinearExplainer...")
        explainer = shap.LinearExplainer(model, X_background)
        shap_values = explainer.shap_values(X_test)
        
        print("✅ LinearExplainer SUCCESS!")
        shap_success = True
        
    except Exception as e:
        print(f"❌ LinearExplainer failed: {str(e)[:100]}")

# Fallback: Dummy values
if not shap_success:
    print("\n⚠️ All SHAP methods failed. Creating dummy values...")
    shap_values = np.zeros((len(X_test), len(feature_names)))
    explainer = None
    print("✅ Dummy SHAP values created (not real SHAP)")

# Save artifacts
try:
    joblib.dump(explainer, '../models/shap_explainer.pkl')
    joblib.dump(shap_values, '../models/shap_values.pkl')
    print("\n✅ SHAP artifacts saved!")
except Exception as e:
    print(f"\n⚠️ Could not save SHAP artifacts: {e}")

# Summary
print("\n" + "="*60)
print("📊 SHAP SUMMARY")
print("="*60)
print(f"SHAP values shape: {shap_values.shape}")
print(f"SHAP success: {'✅' if shap_success else '⚠️ (using dummy)'}")

# Show sample SHAP values
if shap_success:
    sample_idx = 0
    print(f"\nSample SHAP values (first 5 features):")
    for i in range(min(5, len(feature_names))):
        print(f"  {feature_names[i]}: {shap_values[sample_idx][i]:.4f}")

In [ ]:
# Calculate global feature importance
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'SHAP_Importance': np.abs(shap_values).mean(axis=0)
}).sort_values('SHAP_Importance', ascending=False)

# Display top features
print("📊 GLOBAL FEATURE IMPORTANCE")
print("="*60)
print("\nTop 20 features by SHAP importance:")
for idx, row in feature_importance.head(20).iterrows():
    print(f"  {row['Feature']}: {row['SHAP_Importance']:.4f}")

# Save feature importance
feature_importance.to_csv('../models/feature_importance.csv', index=False)
joblib.dump(feature_importance, '../models/feature_importance.pkl')

print("\n✅ Feature importance saved!")
